# 01. TabPFN-2.5의 구조 변경 (1) — 층 수와 셀 3개 묶기

TabPFN-2.5(Grinsztajn et al., arXiv 2511.08667, 2025)는 v2의 셀 단위 2방향 attention 구조를 그대로 쓰고("We follow the alternating-attention transformer design of TabPFNv2"), 논문 3절 "Architecture"에서 네 가지만 바꿨습니다. 층 수를 늘리고, 셀을 3개씩 묶고, 학습되는 "thinking" 행 64개를 표에 붙이고, 회귀 모델의 encoder를 2층 MLP로 바꿨습니다.

이 노트북은 네 가지 가운데 앞의 둘, 층 수와 셀 3개 묶기를 작은 표로 따라갑니다. 뒤의 둘(thinking 행, 회귀 encoder)은 [02 thinking 행과 회귀 MLP encoder](02_thinking_rows_and_regression_encoder.ipynb)에서 봅니다. v2 구조 자체(셀 토큰, feature attention, sample attention)는 [03_tabpfn_v2/01 셀 단위 표현과 2방향 attention](../03_tabpfn_v2/01_cell_tokens_and_two_way_attention.ipynb)에서 다뤘으니, 여기서는 바뀐 부분만 봅니다. 코드는 v2와 같은 `src/minipfn/models/cell_transformer.py`(`CellTransformer`)이고, 2.5는 설정 몇 개만 다릅니다(`src/minipfn/recipes/tabpfn_v2.py`의 `ARCHITECTURES`). v1부터 3.5까지의 차이는 [노트북 로드맵의 버전 비교 표](../README.md#tabpfn-버전-비교)에 모았고, 01·02 노트북은 그 표의 v2 → 2.5 칸(층 수, 셀 묶기, thinking 행, 회귀 encoder)을 따라갑니다.

다루는 흐름:

1. 한눈에 보기 — 설정과 파라미터 수
2. 층 수: 12 → 회귀 18, 분류 24
3. 셀 3개를 토큰 하나로 — feature group 2 → 3
4. 정리

## `minipfn` 불러오기

다른 노트북과 같은 import shim입니다. 이 노트북은 `notebooks/04_tabpfn_v2_5/` 아래에 있으므로 저장소 루트는 두 단계 위입니다.

In [1]:
try:
    import minipfn
except ImportError:
    import sys
    from pathlib import Path

    sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
    import minipfn

In [2]:
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from minipfn import set_seed
from minipfn.models.cell_transformer import CellTransformer, CellTransformerConfig
from minipfn.recipes.tabpfn_v2 import ARCHITECTURES, paper, with_architecture

set_seed(0)  # 재현성을 위해 난수 시드를 고정한다.

## 1. 한눈에 보기 — 설정과 파라미터 수

`ARCHITECTURES`에는 구조마다, task마다 모델 설정이 들어 있습니다. v2는 분류와 회귀가 같은 구조이고, 2.5는 둘이 다릅니다.

In [3]:
rows = {}
for arch, by_task in ARCHITECTURES.items():
    for task, settings in by_task.items():
        rows[f"{arch} {task}"] = settings
display(pd.DataFrame(rows).T.drop(columns="model_type"))

,nlayers,features_per_group,n_thinking_rows,encoder
v2 classification,12,2,0,linear
v2 regression,12,2,0,linear
v2.5 classification,24,3,64,linear
v2.5 regression,18,3,64,mlp


네 열이 이 노트북에서 볼 네 가지입니다. `nlayers`는 층 수(2절), `features_per_group`은 토큰 하나에 묶는 셀 수(3절), `n_thinking_rows`는 thinking 행 수, `encoder`는 셀을 토큰으로 바꾸는 층입니다(뒤의 둘은 [02 노트북](02_thinking_rows_and_regression_encoder.ipynb)). 2.5 논문은 임베딩 크기(emsize)와 head 수를 적지 않아서, 그 둘은 v2(192, 6) 그대로 둡니다.

이 설정으로 실제 크기의 모델을 만들어 파라미터 수를 부분별로 셉니다. 학습 설정 `paper(task)`에 `with_architecture(..., arch)`로 구조를 입힌 뒤 `build_model()`을 부르면 학습에 쓰는 모델과 같은 것이 만들어집니다.

In [4]:
counts = {}
for arch in ["v2", "v2.5"]:
    for task in ["classification", "regression"]:
        model = with_architecture(paper(task), arch).build_model()
        parts = {}
        for name, p in model.named_parameters():
            part = name.split(".")[0]  # 맨 앞 이름: layers, decoder, thinking_rows, ...
            parts[part] = parts.get(part, 0) + p.numel()
        parts["합계"] = model.num_parameters()
        counts[f"{arch} {task}"] = parts

order = [
    "feature_embedder",
    "target_embedder",
    "column_embedder",
    "thinking_rows",
    "layers",
    "decoder",
    "합계",
]
param_table = pd.DataFrame(counts).reindex(order).fillna(0).astype(int)
display(param_table.style.format("{:,}"))

,v2 classification,v2 regression,v2.5 classification,v2.5 regression
feature_embedder,960,960,"1,344","38,400"
target_embedder,576,576,576,576
column_embedder,"9,408","9,408","9,408","9,408"
thinking_rows,0,0,"12,288","12,288"
layers,"7,077,888","7,077,888","14,155,776","10,616,832"
decoder,"155,914","3,993,224","155,914","3,993,224"
합계,"7,244,746","11,082,056","14,335,306","14,670,728"


파라미터 대부분은 층(`layers`)에 있습니다. 분류 모델은 층이 12개에서 24개로 두 배가 되어 723만 개에서 1,434만 개로 거의 두 배가 됩니다. 회귀 모델은 층이 18개라 1.5배이고, 출력층(`decoder`)이 bar distribution 구간 5,000개의 logit을 내느라 원래 커서 전체로는 1.3배입니다. thinking 행은 12,288개(= 64행 × 192차원)로 작고, 회귀의 MLP encoder는 feature encoder를 960개에서 38,400개로 늘립니다. v2.5 분류의 `feature_embedder`가 960개에서 1,344개로 늘어난 것은 셀 3개를 묶어 입력이 커졌기 때문입니다(3절).

## 2. 층 수: 12 → 회귀 18, 분류 24

논문은 "We increase the network depth from 12 to 18 layers for our regression model and 24 layers for our classification model"이라고만 적었습니다. 층 하나의 구조는 v2와 같습니다: feature attention → sample attention → MLP, 각 단계 뒤에 잔차와 LayerNorm. 층 하나의 파라미터를 손으로 세어 봅니다. 임베딩 크기를 $E = 192$라고 하면 다음과 같습니다.

In [5]:
E = 192
attention = 3 * E * E + E * E  # query·key·value 투영 (E → 3E) + 출력 투영 (E → E), bias 없음
mlp = E * (4 * E) + (4 * E) * E  # E → 4E → E, bias 없음
per_layer = {
    "feature attention": attention,
    "sample attention": attention,
    "MLP": mlp,
}
per_layer["층 하나"] = sum(per_layer.values())
display(pd.Series(per_layer, name="파라미터 수").to_frame().style.format("{:,}"))

model_layer = sum(
    p.numel() for p in CellTransformer(CellTransformerConfig()).layers[0].parameters()
)
print(f"실제 모델의 층 하나: {model_layer:,}")

,파라미터 수
feature attention,"147,456"
sample attention,"147,456"
MLP,"294,912"
층 하나,"589,824"


실제 모델의 층 하나: 589,824


층 하나는 약 59만 개($16E^2$)이고, 12층이면 708만 개, 24층이면 1,416만 개입니다. 1절 표의 `layers` 값과 같습니다.

층을 늘리면 계산량도 층 수에 비례해 늘어납니다. 다만 층 하나의 계산량은 토큰 수에 달려 있고, 3절의 셀 묶기가 토큰 수를 줄여 이 비용 일부를 되돌립니다.

## 3. 셀 3개를 토큰 하나로 — feature group 2 → 3

v2는 이웃한 feature 셀 2개를 묶어 토큰 하나로 만들었고, 2.5는 3개를 묶습니다("We use a group size of 3 for TabPFN-2.5, compared to 2 for TabPFNv2"). v2 primer의 과일 표에 feature를 셋 더해 5개로 만든 표로 따라갑니다. 학습 행 4개(사과 2, 배 2)와 정답을 가린 테스트 행 2개입니다.

In [6]:
fruit = pd.DataFrame(
    {
        "무게(g)": [210, 190, 320, 290, 200, 310],
        "당도(brix)": [13.5, 14.0, 11.0, 11.5, 13.0, 12.0],
        "지름(cm)": [7.5, 7.2, 9.0, 8.6, 7.4, 8.8],
        "산도(pH)": [3.4, 3.5, 3.9, 3.8, 3.4, 3.9],
        "껍질(mm)": [0.6, 0.5, 1.0, 0.9, 0.6, 1.0],
    }
)
y = np.array([0, 0, 1, 1])  # 학습 행의 정답: 0 = 사과, 1 = 배
n_train = 4
roles = ["학습 1", "학습 2", "학습 3", "학습 4", "테스트 5", "테스트 6"]

shown = fruit.copy()
shown["y"] = ["0 (사과)", "0 (사과)", "1 (배)", "1 (배)", "?", "?"]
shown.index = roles
display(shown)

,무게(g),당도(brix),지름(cm),산도(pH),껍질(mm),y
학습 1,210,13.5,7.5,3.4,0.6,0 (사과)
학습 2,190,14.0,7.2,3.5,0.5,0 (사과)
학습 3,320,11.0,9.0,3.9,1.0,1 (배)
학습 4,290,11.5,8.6,3.8,0.9,1 (배)
테스트 5,200,13.0,7.4,3.4,0.6,?
테스트 6,310,12.0,8.8,3.9,1.0,?


**1단계: 열마다 표준화합니다.** v2와 같습니다. 평균과 표준편차는 학습 행 4개에서만 구합니다. 모델 안(`CellTransformer._embed_features`)에서 하는 계산이라 입력에는 원래 값을 그대로 넣습니다.

In [7]:
features = fruit.to_numpy(dtype=float)
mean = features[:n_train].mean(axis=0)
std = features[:n_train].std(axis=0)  # 모델과 같이 n으로 나눈 표준편차
z = (features - mean) / std

display(pd.DataFrame(z, columns=fruit.columns, index=roles).round(2))

,무게(g),당도(brix),지름(cm),산도(pH),껍질(mm)
학습 1,-0.79,0.78,-0.77,-1.21,-0.73
학습 2,-1.16,1.18,-1.17,-0.73,-1.21
학습 3,1.25,-1.18,1.24,1.21,1.21
학습 4,0.69,-0.78,0.70,0.73,0.73
테스트 5,-0.97,0.39,-0.90,-1.21,-0.73
테스트 6,1.06,-0.39,0.97,1.21,1.21


**2단계: 이웃한 열을 `g`개씩 묶습니다.** feature 5개를 2개씩 묶으면 3묶음이고, 마지막 묶음은 한 칸이 빕니다. 3개씩 묶으면 2묶음이고, 역시 마지막 묶음이 한 칸 빕니다. 빈 칸은 0으로 채웁니다. 첫 학습 행으로 보면 다음과 같습니다.

In [8]:
def group_row(values, g):
    # 열 수를 g의 배수로 맞추려고 뒤에 0을 채운 뒤 g개씩 자른다
    pad = -len(values) % g
    padded = np.concatenate([values, np.zeros(pad)])
    return padded.reshape(-1, g)


row = z[0]  # 첫 학습 행의 표준화한 값 5개
for g in [2, 3]:
    groups = group_row(row, g)
    print(f"g = {g}: 묶음 {len(groups)}개")
    display(
        pd.DataFrame(
            groups.round(2),
            index=[f"묶음 {i + 1}" for i in range(len(groups))],
            columns=[f"칸 {j + 1}" for j in range(g)],
        )
    )

g = 2: 묶음 3개


,칸 1,칸 2
묶음 1,-0.79,0.78
묶음 2,-0.77,-1.21
묶음 3,-0.73,0.00


g = 3: 묶음 2개


,칸 1,칸 2,칸 3
묶음 1,-0.79,0.78,-0.77
묶음 2,-1.21,-0.73,0.00


**3단계: 빈 칸이 있는 묶음은 크기를 키웁니다.** 묶음의 숫자 $g$개 가운데 실제 열이 $k$개뿐이면, 그 묶음에 $\sqrt{g / k}$를 곱합니다. 묶음의 크기(제곱합)가 채워진 칸 수에 덜 휘둘리게 하려는 것입니다. g = 2의 묶음 3은 $\sqrt{2/1} \approx 1.41$배, g = 3의 묶음 2는 $\sqrt{3/2} \approx 1.22$배가 됩니다.

In [9]:
def group_row_scaled(values, g):
    groups = group_row(values, g)
    used = group_row(np.ones_like(values), g).sum(axis=1, keepdims=True)  # 묶음마다 실제 열 수 k
    return groups * np.sqrt(g / used)


for g in [2, 3]:
    print(f"g = {g}")
    display(
        pd.DataFrame(
            group_row_scaled(row, g).round(2),
            index=[f"묶음 {i + 1}" for i in range(-(-len(row) // g))],
            columns=[f"칸 {j + 1}" for j in range(g)],
        )
    )

g = 2


,칸 1,칸 2
묶음 1,-0.79,0.78
묶음 2,-0.77,-1.21
묶음 3,-1.03,0.00


g = 3


,칸 1,칸 2,칸 3
묶음 1,-0.79,0.78,-0.77
묶음 2,-1.49,-0.89,0.00


**4단계: 묶음마다 [값 g개, 결측 표시 g개]를 encoder에 넣어 토큰 하나를 만듭니다.** 결측 표시는 v2처럼 결측값이면 $-2$, 아니면 $0$입니다. 묶음을 채우려고 넣은 빈 칸은 결측값이 아니라서 표시도 $0$입니다. 모델이 빈 칸을 알아보는 단서는 3단계의 배율뿐입니다. encoder의 입력 크기는 v2가 4, 2.5가 6입니다.

손 계산이 실제 모델과 같은지 확인합니다. 작은 `CellTransformer`를 두 묶음 크기로 만들고, encoder(`feature_embedder`)에 들어가는 텐서를 hook으로 꺼내 봅니다.

In [10]:
x = torch.tensor(features, dtype=torch.float32)[None]  # (데이터셋 1, 행 6, feature 5)
y_train = torch.tensor(y, dtype=torch.float32)[None]  # (데이터셋 1, 학습 행 4)

for g in [2, 3]:
    tiny = CellTransformer(
        CellTransformerConfig(emsize=16, nlayers=1, nhead=2, features_per_group=g, n_out=2)
    )
    captured = {}
    hook = tiny.feature_embedder.register_forward_hook(
        lambda module, inputs, output, captured=captured: captured.update(x=inputs[0], token=output)
    )
    with torch.no_grad():
        tiny(x, y_train)
    hook.remove()

    encoder_input = captured["x"][0, 0]  # 첫 학습 행: (묶음 수, 2g)
    values = encoder_input[:, :g].numpy()  # 앞 g개가 값
    indicators = encoder_input[
        :, g:
    ].numpy()  # 뒤 g개가 결측 표시 (결측값이 없으니 빈 칸까지 모두 0)
    same = np.allclose(values, group_row_scaled(row, g), atol=1e-5) and not indicators.any()
    in_shape, token_shape = tuple(captured["x"].shape), tuple(captured["token"].shape)
    print(f"g = {g}: encoder 입력 {in_shape}, 토큰 {token_shape}, 손 계산과 같음: {same}")

g = 2: encoder 입력 (1, 6, 3, 4), 토큰 (1, 6, 3, 16), 손 계산과 같음: True
g = 3: encoder 입력 (1, 6, 2, 6), 토큰 (1, 6, 2, 16), 손 계산과 같음: True


encoder 입력의 모양은 (데이터셋, 행, 묶음 수, 2g)이고, 토큰은 (데이터셋, 행, 묶음 수, E)입니다. 손 계산과 실제 모델이 같습니다. 행마다 토큰 수는 v2가 3개, 2.5가 2개이고, 여기에 y 토큰이 하나씩 붙습니다.

**토큰 수가 줄면 계산이 줄어듭니다.** 행 하나의 토큰 수를 $C$라고 하면 feature 묶음 수에 y 토큰 하나를 더한 $C = \lceil F / g \rceil + 1$입니다. feature attention은 행마다 토큰 $C$개가 서로를 보므로 $C^2$에, sample attention과 MLP는 $C$에 비례합니다. feature 수별로 세어 봅니다.

In [11]:
token_rows = {}
for n_features in [5, 30, 100, 500, 2000]:
    c_v2 = -(-n_features // 2) + 1  # 올림(F / 2) + y 토큰
    c_v25 = -(-n_features // 3) + 1
    token_rows[n_features] = {
        "v2 토큰 수 C": c_v2,
        "2.5 토큰 수 C": c_v25,
        "C 비율 (2.5 / v2)": c_v25 / c_v2,
        "C² 비율 (feature attention)": (c_v25 / c_v2) ** 2,
    }
token_table = pd.DataFrame(token_rows).T.rename_axis("feature 수")
display(token_table.astype({"v2 토큰 수 C": int, "2.5 토큰 수 C": int}).round(2))

,v2 토큰 수 C,2.5 토큰 수 C,C 비율 (2.5 / v2),C² 비율 (feature attention)
feature 수,,,,
5,4,3,0.75,0.56
30,16,11,0.69,0.47
100,51,35,0.69,0.47
500,251,168,0.67,0.45
2000,1001,668,0.67,0.45


feature가 많으면 토큰 수는 2/3, feature attention의 계산은 4/9로 줄어듭니다. 층 하나의 계산이 대략 2/3이 되니, 층을 24개로 두 배 늘린 분류 모델은 v2의 2배가 아니라 1.3~1.4배입니다(2 × 2/3). 여기에 thinking 행 64개([02 노트북](02_thinking_rows_and_regression_encoder.ipynb) 1절)가 행 수를 늘립니다. 아래 표(행 640개)에서는 704행이 되어 1.1배쯤 더해지므로, 분류 모델은 v2의 1.5배 안팎이 됩니다. 18층 회귀 모델은 같은 계산으로 v2의 1.1배쯤입니다.

실제 크기 모델의 forward 시간을 CPU에서 재 봅니다. 학습 행 512개와 테스트 행 128개짜리 분류 표입니다. thinking 행 64개도 행 수를 늘리므로 함께 들어갑니다. 다섯 번 재서 가운데 값을 씁니다.

In [12]:
def forward_seconds(model, x, y_train, repeat=5):
    with torch.inference_mode():
        model(x, y_train)  # 처음 한 번은 준비(캐시)로 빼고 잰다
        seconds = []
        for _ in range(repeat):
            start = time.perf_counter()
            model(x, y_train)
            seconds.append(time.perf_counter() - start)
    return float(np.median(seconds))  # 여러 번 재서 가운데 값


torch.manual_seed(0)
models = {
    arch: with_architecture(paper("classification"), arch).build_model().eval()
    for arch in ["v2", "v2.5"]
}
timing = {}
for n_features in [30, 120]:
    x_big = torch.randn(1, 512 + 128, n_features)
    y_big = torch.randint(0, 2, (1, 512)).float()
    seconds = {arch: forward_seconds(m, x_big, y_big) for arch, m in models.items()}
    timing[n_features] = {
        "v2 (12층) 초": seconds["v2"],
        "2.5 (24층) 초": seconds["v2.5"],
        "비율": seconds["v2.5"] / seconds["v2"],
    }
display(pd.DataFrame(timing).T.rename_axis("feature 수").round(2))

,v2 (12층) 초,2.5 (24층) 초,비율
feature 수,,,
30,0.6,0.95,1.60
120,1.9,2.94,1.55


층이 두 배인데 시간은 1.5~1.6배로, 위에서 어림한 1.5배와 비슷하게 늘어납니다. CPU에서 잰 값이라 실행마다 조금씩 다르고, GPU와도 다를 수 있습니다. 논문은 "TabPFN-2.5 is between 1x and 2.3x faster thanks to optimized preprocessing and larger feature groups"라고 적었는데, 이 숫자는 전처리까지 포함한 추론 시간이라 모델 forward만 잰 위 표와 직접 비교할 수 없습니다.

## 4. 정리

| 바뀐 것 | v2 | 2.5 | 이 노트북 |
|---|---|---|---|
| 층 수 | 12 | 회귀 18, 분류 24 | 2절 |
| 셀 묶음 크기 | 2 | 3 (토큰 수는 feature가 많으면 2/3) | 3절 |

층이 늘어 파라미터가 분류 723만 → 1,434만으로 거의 두 배가 되고, 셀 3개 묶기는 토큰 수를 줄여 층 하나의 계산을 2/3쯤으로 되돌립니다. 다음 노트북 [02 thinking 행과 회귀 MLP encoder](02_thinking_rows_and_regression_encoder.ipynb)는 나머지 두 가지, thinking 행 64개와 회귀 모델의 2층 MLP encoder를 보고 네 가지를 한 표로 정리합니다.